# Cartographic furniture & styling (3-D)

The 3-D (PyVista) tier draws the same cartographic furniture a 2-D map does, plus the lighting and export a
true-3-D scene needs. This notebook walks through the verbs that frame, annotate, light and export a scene —
`domain`, `graticule`, `scale_bar`, `north_arrow`, `attribution`, `labels`, `sun`/`shadows`/`ambient_occlusion`,
`bivariate`, `basemap`, `export_vector` and scene bundles — each explained before it is shown.

Every scene here is built **off-screen** (headless), with synthetic data, so the notebook runs anywhere with no
files and, where a step needs the network, guards it.

In [ ]:
import numpy as np

from digitalearth.base.sources import get_source
from digitalearth.three_d import Scene3D

# A synthetic geographic relief over Europe, so the framing and the graticule have lon/lat to work with.
lon = np.linspace(-10.0, 30.0, 40)
lat = np.linspace(35.0, 60.0, 30)
relief = np.add.outer(np.cos(np.radians(lat)), np.sin(np.radians(lon)))
source = get_source(relief, x=lon, y=lat)

scene = Scene3D(off_screen=True, crs=4326)
scene.terrain(source, z_exaggeration=8.0)
scene.screenshot().shape

## Framing — `domain`

`domain` points the camera at a named region or an explicit `(west, south, east, north)` box, reprojected into
the scene's display CRS through pyramids. It frames the view without flattening the relief.

In [ ]:
scene.domain("europe")
scene.screenshot().shape

## Graticule — `graticule`

A latitude/longitude grid, labelled along its edges. `lon_step`/`lat_step` (or `spacing` for a square grid) set
the interval; the lines are reprojected into the display CRS so the grid lies on the data.

In [ ]:
scene.graticule(spacing=10.0)
scene.layer_ids

## Scale bar, north arrow, attribution

A ground-distance scale bar (which switches the view to orthographic — the only projection it is honest in), a
camera-aware north arrow, and a corner credit line, so a rendered scene carries its own context.

In [ ]:
scene.scale_bar()
scene.north_arrow()
scene.attribution("© Natural Earth")
scene.screenshot().shape

## Labels — `labels`

Label a feature collection from one of its columns, each label on a filled halo so it reads against the scene.

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

cities = gpd.GeoDataFrame(
    {"name": ["Paris", "Berlin", "Madrid"]},
    geometry=[Point(2.35, 48.85), Point(13.4, 52.52), Point(-3.7, 40.42)],
    crs=4326,
)
scene.labels(cities, "name")
scene.layer_ids

## Lighting — `sun`, `shadows`, `ambient_occlusion`

Rasterization-grade relief shading: a directional sun placed by azimuth and elevation, plus cast shadows and
ambient occlusion. (This is not path tracing — VTK is a rasterizer — but it gives relief real depth.)

In [ ]:
scene.sun(azimuth=135.0, elevation=25.0)
scene.shadows()
scene.ambient_occlusion()
scene.screenshot().shape

## Bivariate styling — `bivariate`

Two variables encoded as one colour from an n×n matrix: each variable is cut into classes by the shared
classifier, and the class pair picks a colour. The classification is engine-neutral; the tier drapes the
resulting image on a ground plane.

In [ ]:
a = np.add.outer(np.linspace(0.0, 1.0, 30), np.linspace(0.0, 1.0, 40))
b = np.add.outer(np.linspace(1.0, 0.0, 30), np.linspace(0.0, 1.0, 40))
bivariate_scene = Scene3D(off_screen=True)
bivariate_scene.bivariate(a, b)
shape = bivariate_scene.screenshot().shape
bivariate_scene.close()
shape

## Basemap — `basemap`

Web-map tiles draped under the scene as ground cover. Tiles are Web Mercator, so draw the scene in EPSG:3857
for them to line up. This cell reaches the network, so it is guarded — it renders when a tile server is
reachable and reports a skip otherwise.

In [ ]:
mercator = Scene3D(off_screen=True, crs=3857)
try:
    mercator.basemap("CartoDB.Positron", bbox=(-10.0, 35.0, 30.0, 60.0), zoom=3)
    print("basemap drawn:", mercator.layer_ids)
except Exception as error:  # offline, or the tile server is unreachable
    print("basemap skipped (no network):", type(error).__name__)
finally:
    mercator.close()

## Export — vector graphics and scene bundles

`export_vector` writes the view as SVG/PDF/EPS through VTK's GL2PS exporter (a high-resolution raster master is
`screenshot(scale=N)`). `save_bundle` writes the whole scene to a versioned zip — but a bundle stores a
*description*, so it needs path- or URL-backed data; a scene built from in-memory arrays is refused, which this
cell shows.

In [ ]:
import os
import tempfile

folder = tempfile.mkdtemp()
scene.export_vector(os.path.join(folder, "scene.svg"))
print("vector export written:", os.path.exists(os.path.join(folder, "scene.svg")))

try:
    scene.save_bundle(os.path.join(folder, "scene.dez"))
except ValueError as error:
    print("in-memory scene is not bundleable:", str(error).split(";")[0][:72], "...")

## One-call framing — `quickmap(backend="3d")`

`quickmap` wires the framing (and, for an EPSG:3857 scene, the basemap) into one call. Guarded, since the
input shape and any basemap fetch depend on the environment.

In [ ]:
from digitalearth import quickmap

try:
    quick = quickmap(relief, backend="3d", domain="europe")
    print("quickmap 3-D layers:", quick.layer_ids)
    quick.close()
except Exception as error:
    print("quickmap demo skipped:", type(error).__name__)

scene.close()